In [15]:
import pandas as pd

import src
from src.load import DataLoader

In [16]:
pd.set_option("display.max_rows", 1024)
pd.set_option("display.max_colwidth", 256)

In [17]:
dl = DataLoader()

In [18]:
videos = (
    dl.videos(filtered=True)
    .join(dl.channels(), "channel_id")
    .select(
        ["video_id", "channel", "video_likes", "video_views", "video_uploadtime", "video_title"],
    )
    .to_pandas()
)

sents = dl.sentences(filtered=True).join(dl.popbert(filtered=True), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sentences=("video_id", "size"),
    elite=("elite", "mean"),
    pplcentr=("pplcentr", "mean"),
)

videos = videos.merge(sents, on="video_id")

# Most Liked

In [19]:
top_like_videos = videos[videos.channel != "FDP"]

quantiles = top_like_videos.groupby("channel").video_likes.quantile(q=0.99).rename("quantile")

In [20]:
df = top_like_videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.video_likes > x["quantile"] else 0, axis=1)

In [21]:
top_videos = (
    df.sort_values(["channel", "video_likes"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [24]:
top_videos.reset_index()[["channel", "video_likes", "video_views", "video_title"]].to_csv(
    src.OUT / "tables/most_liked_videos_per_channel.csv",
    index=False,
)

# Most Viewed

In [ ]:
quantiles = top_videos.groupby("channel").video_views.quantile(q=0.99).rename("quantile")

In [ ]:
df = top_like_videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.video_views > x["quantile"] else 0, axis=1)

In [ ]:
top_videos = (
    df.sort_values(["channel", "video_views"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [ ]:
top_videos.reset_index()[["channel", "video_likes", "video_views", "video_title"]].to_csv(
    src.OUT / "tables/most_viewed_videos_per_channel.csv",
    index=False,
)

# Most Anti-Elitism

In [25]:
quantiles = videos.groupby("channel").elite.quantile(q=0.99).rename("quantile")

In [26]:
df = videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.elite > x["quantile"] else 0, axis=1)

In [27]:
top_videos = (
    df.sort_values(["channel", "elite"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [29]:
top_videos.reset_index()[["channel", "n_sentences", "elite", "video_title"]].to_csv(
    src.OUT / "tables/most_antielitism_videos_per_channel.csv",
    index=False,
)